# Model 2 (CNN+GRU) training — Ask the Sensors

Trains the CNN+GRU recognizer (PLAN.md Step 10, Model 2) on GPU, using the exact same code that lives in the project's GitHub repo and the exact same official 5-fold split and already-verified data as Model 1 (TinyCNN) and the Random Forest — so the results are directly, fairly comparable.

**Before running anything below:**
1. Open the notebook's right-hand settings panel → **Accelerator** → pick **GPU T4 x2** (or P100 if that's what's offered).
2. In the same panel → **Add Input** → search for `ubiquitous-comp-raw-minutes` → add it (you said you can already see it in your account, so this should just be selecting it).
3. Internet must be **On** (needed to `git clone` the repo) — also in that settings panel.

Run the cells top to bottom. Cell 5 is a quick smoke test — if it crashes, the real error will show up directly below that cell (not hidden the way it was in the batch runs), so just share whatever error you see and it'll be a two-minute fix instead of another guessing round.

## Step 0 — install a GPU-compatible PyTorch build

Kaggle's pre-installed PyTorch can be too new for whichever GPU you get assigned (this already happened once: a Tesla P100 was rejected by the default build). `2.2.2+cu118` is confirmed to work with Kaggle's Python version and is old enough to support P100/T4-class GPUs.

In [ ]:
!pip install -q torch==2.2.2 --index-url https://download.pytorch.org/whl/cu118

import torch
print("torch:", torch.__version__, " cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.zeros(1, device="cuda") + 1
    print("GPU tensor op OK")
else:
    print("WARNING: no GPU detected -- check the accelerator setting in the right panel")

## Step 1 — get the project code

Clones the actual GitHub repo fresh, so this notebook always runs the real, current version of the training code — not a copy that could drift out of sync.

In [ ]:
import subprocess, os

REPO_DIR = "/kaggle/working/repo"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/chadsaras/Ubiquitous_Comp.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("working directory:", os.getcwd())

## Step 2 — locate the attached dataset

Finds `raw_minutes.npz` and the official fold-membership files wherever Kaggle actually mounted them (searched, not assumed — the exact mount path isn't always what you'd expect), then stages the fold files into the folder layout the training code expects.

In [ ]:
import glob, shutil, zipfile

npz_matches = glob.glob("/kaggle/input/**/raw_minutes.npz", recursive=True)
assert npz_matches, "raw_minutes.npz not found under /kaggle/input -- is the dataset attached?"
raw_path = npz_matches[0]
print("raw_minutes.npz:", raw_path, " size MB:", round(os.path.getsize(raw_path) / 1e6))

fold_txts = glob.glob("/kaggle/input/**/fold_0_train_android_uuids.txt", recursive=True)
if not fold_txts:
    for zpath in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        print("extracting", zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall("/kaggle/working/extracted")
    fold_txts = glob.glob("/kaggle/working/extracted/**/fold_0_train_android_uuids.txt", recursive=True)
assert fold_txts, "no fold_*.txt files found even after checking for a zip -- see the /kaggle/input listing below"

data_dir = "/kaggle/working/data_for_folds"
cv5 = os.path.join(data_dir, "raw", "cv5Folds")
os.makedirs(cv5, exist_ok=True)
for src in glob.glob(os.path.dirname(fold_txts[0]) + "/fold_*.txt"):
    shutil.copy(src, cv5)
print("staged", len(os.listdir(cv5)), "fold files into", cv5)
print("\nraw_path =", raw_path)
print("data_dir =", data_dir)

If either assertion above fails, run this to see what Kaggle actually mounted, then fix the path by hand:
```python
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))
```

## Step 3 — sanity check: does the real model run one real training step on this GPU?

This is exactly the step that was failing silently in the batch runs. Run it here and whatever happens will be visible directly below — a clean "backward OK", or a real traceback we can actually read and fix.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from src.recognize.nn_models import CNNGRU, count_params

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNGRU().to(device)
print("params:", f"{count_params(model):,}", " device:", device)

dummy = torch.randn(4, 6, 500, device=device)
out = model(dummy)
print("forward OK, output shape:", tuple(out.shape))

loss = out.sum()
loss.backward()
print("backward OK -- a real training step works on this GPU")

## Step 4 — smoke test: 1 fold, 2 epochs

Confirms the whole pipeline (data loading, fold splitting, training loop, evaluation, saving) works end to end before committing to the full ~2 hour run. Output streams live below.

In [ ]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
!python -u scripts/train_cnn.py --arch cnngru --device {dev} --raw {raw_path} --data-dir {data_dir} \
    --folds 0 --epochs 2 --patience 1

If the cell above finished and printed accuracy / macro-F1 numbers, everything works — move on to the full run. If it crashed, the traceback is right there above; that tells us exactly what to fix next.

## Step 5 — the real run: all 5 folds + final model

This is the one that produces the actual comparable results (`results/classifier_metrics_cnngru.json`, `results/confusion_matrix_cnngru.png`, `results/model_cnngru.pt`). Expect this to take a while — watch the live epoch-by-epoch output below.

In [ ]:
!python -u scripts/train_cnn.py --arch cnngru --device {dev} --raw {raw_path} --data-dir {data_dir}

## Step 6 — check what got produced

In [ ]:
for f in sorted(os.listdir("results")):
    if "cnngru" in f:
        path = os.path.join("results", f)
        print(f"{f:35s} {os.path.getsize(path)/1e6:8.3f} MB")

## Step 7 — get the results back into the project repo

**Easiest option:** open the **Output** panel on the right side of this notebook, find `results/model_cnngru.pt`, `results/classifier_metrics_cnngru.json`, and `results/confusion_matrix_cnngru.png`, download them, and hand them over (or send me the numbers) — I'll commit them into the repo the same way the institute-server results were committed.

**Automatic option** (optional): push straight back to GitHub from this notebook.
1. Create a GitHub personal access token with `repo` scope at github.com/settings/tokens.
2. In this notebook: **Add-ons → Secrets** → add a secret named `GITHUB_TOKEN` with that value.
3. Uncomment and run the cell below.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add results/*cnngru*
# !git commit -m "Add Model 2 (CNN+GRU) training results from Kaggle GPU"
# !git remote set-url origin https://{token}@github.com/chadsaras/Ubiquitous_Comp.git
# !git push origin main